In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

combine = [train, test]

for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map({'male': 0, 'female': 1})
    dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
    dataset['Fare'] = dataset['Fare'].fillna(dataset['Fare'].median())
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1
    dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+).', expand=False)
    dataset['Title'] = dataset['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
        'Rare'
    )
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    dataset['Title'] = dataset['Title'].map({
        'Mr': 1,
        'Miss': 2,
        'Mrs': 3,
        'Master': 4,
        'Rare': 5
    })
    dataset['Title'] = dataset['Title'].fillna(0)

features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'FamilySize',
    'IsAlone',
    'Title'
]

X_train = train[features]
y_train = train['Survived']
X_test = test[features]

xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

scores = cross_val_score(xgb, X_train, y_train, cv=5)
print("Cross Validation Accuracy:", scores.mean())

xgb.fit(X_train, y_train)

predictions = xgb.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()


Cross Validation Accuracy: 0.8495825748540581


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
